In [5]:
import warnings
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error

# --- Global Constants ---
PREDS_PATH = './model_predictions/'
DATA_PATH = './'
OLD_BEST_SCORE = 296350.26 # The score from your winning notebook
N_OPTUNA_TRIALS = 30
COMPETITION_ALPHA = 0.1
RANDOM_STATE = 42

# --- Winkler Score Helper Function (Upgraded) ---
def winkler_score(y_true, lower, upper, alpha=COMPETITION_ALPHA, return_coverage=False):
    """
    Upgraded function to also return coverage and width for diagnostics.
    """
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (lower - y_true) * (2 / alpha), 0)
    penalty_upper = np.where(y_true > upper, (y_true - upper) * (2 / alpha), 0)
    score = np.mean(width + penalty_lower + penalty_upper)
    
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return score, coverage, np.mean(width)
    return score

warnings.filterwarnings('ignore')
print("Libraries and helper functions loaded successfully.")

Libraries and helper functions loaded successfully.


In [6]:
print("--- Loading all pre-trained base model predictions ---")

try:
    # --- Load Mean Model Predictions ---
    oof_xgb_preds = np.load(f'{PREDS_PATH}oof_xgb_preds.npy')
    test_xgb_preds = np.load(f'{PREDS_PATH}test_xgb_preds.npy')
    oof_cb_preds = np.load(f'{PREDS_PATH}oof_cb_preds.npy')
    test_cb_preds = np.load(f'{PREDS_PATH}test_cb_preds.npy')
    oof_nn_preds = np.load(f'{PREDS_PATH}oof_nn_preds.npy')
    test_nn_preds = np.load(f'{PREDS_PATH}test_nn_preds.npy')
    print("All MEAN model predictions loaded.")
    
    # --- Load Ground Truth Target ---
    y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
    print("Ground truth target 'y_true' loaded.")

except FileNotFoundError as e:
    print(f"\nERROR: Could not find a prediction file. {e}")
    print("Please ensure you have run all training notebooks and saved their predictions first.")

print("\nAll predictions loaded successfully. Ready to build the final ensemble.")

--- Loading all pre-trained base model predictions ---
All MEAN model predictions loaded.
Ground truth target 'y_true' loaded.

All predictions loaded successfully. Ready to build the final ensemble.


In [7]:
print("\n--- STAGE 1: Finding Optimal Weights for the 3-Model MEAN Ensemble ---")

# Stack the OOF predictions into a single array for the optimizer
oof_preds_stack_mean = np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T

# Define the objective function (RMSE) for the mean ensemble
def get_ensemble_rmse(weights):
    final_prediction = np.dot(oof_preds_stack_mean, weights)
    return np.sqrt(mean_squared_error(y_true, final_prediction))

# Run the optimizer to find the best blending weights
result_mean = minimize(get_ensemble_rmse, [1/3]*3, method='SLSQP',
                       bounds=[(0,1)]*3,
                       constraints=({'type': 'eq', 'fun': lambda w: 1 - sum(w)}))

best_mean_weights = result_mean.x
best_ensemble_rmse = result_mean.fun

# Create the final blended mean predictions using the optimal weights
oof_ensemble_mean = np.dot(oof_preds_stack_mean, best_mean_weights)
test_ensemble_mean = np.dot(np.vstack([test_xgb_preds, test_cb_preds, test_nn_preds]).T, best_mean_weights)

print("\n" + "="*50)
print("      OPTIMAL 3-MODEL MEAN BLEND SEARCH COMPLETE")
print("="*50)
print(f"Lowest Ensemble RMSE achieved: ${best_ensemble_rmse:,.2f}")
print(f"Optimal Weight for XGBoost : {best_mean_weights[0]:.4f}")
print(f"Optimal Weight for CatBoost: {best_mean_weights[1]:.4f}")
print(f"Optimal Weight for NeuralNet: {best_mean_weights[2]:.4f}")
print("="*50)
print("\nSuper-ensemble mean predictions are now created and ready for use as meta-features.")


--- STAGE 1: Finding Optimal Weights for the 3-Model MEAN Ensemble ---

      OPTIMAL 3-MODEL MEAN BLEND SEARCH COMPLETE
Lowest Ensemble RMSE achieved: $96,778.10
Optimal Weight for XGBoost : 0.4100
Optimal Weight for CatBoost: 0.4779
Optimal Weight for NeuralNet: 0.1121

Super-ensemble mean predictions are now created and ready for use as meta-features.


In [20]:
print("\n--- STAGE 2: Tuning the ENHANCED Quantile Stacking Meta-Model ---")

# --- Load Original Features for Context ---
# We need to give the meta-model more context about the properties.
# We will use a lightweight version of the feature engineering from your other notebooks.
print("Loading original raw data for feature engineering...")
df_train_raw = pd.read_csv(DATA_PATH + 'dataset.csv')
# Select only the most powerful, non-leaky features
context_features = [
    'grade', 'sqft', 'lat', 'long', 'yr_built', 'yr_renovated' 
]
# For simplicity, we will assume these column names. Adjust if yours are different.
# Simple renaming for consistency if needed:
df_train_raw = df_train_raw.rename(columns={'latitude': 'lat', 'longitude': 'long', 'year_built': 'yr_built', 'year_reno':'yr_renovated'})
X_context = df_train_raw[['grade', 'sqft', 'lat', 'long', 'yr_built', 'yr_renovated']].fillna(0)
X_context['property_age'] = 2025 - X_context['yr_built']


# --- Create the ENHANCED Feature Set for the Meta-Model ---
# We combine the base model predictions with the powerful context features.
X_meta_train_enhanced = pd.concat([
    pd.DataFrame({
        'xgb_pred': oof_xgb_preds,
        'cb_pred': oof_cb_preds,
        'nn_pred': oof_nn_preds,
        'ensemble_mean': oof_ensemble_mean
    }),
    X_context.reset_index(drop=True) # Ensure indices align
], axis=1)


print(f"Enhanced meta-model training data created with shape: {X_meta_train_enhanced.shape}")

# --- Create a validation set for tuning ---
from sklearn.model_selection import train_test_split
X_meta_train_tune, X_meta_val_tune, y_meta_train_tune, y_meta_val_tune = train_test_split(
    X_meta_train_enhanced, y_true, test_size=0.25, random_state=RANDOM_STATE
)
N_OPTUNA_TRIALS = 100
# --- Define the Optuna Objective Function ---
def objective_winkler_meta_enhanced(trial):
    params = {
        'objective': 'reg:quantileerror', 'tree_method': 'hist', 'n_jobs': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.025),  # Narrowed around best 0.02
        'max_depth': trial.suggest_int('max_depth', 5, 7),  # Best values clustered here
        'subsample': trial.suggest_float('subsample', 0.75, 0.92),  # Optimal range 0.75-0.92
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.78, 0.95),  # Higher values better
        'lambda': trial.suggest_float('lambda', 5.0, 15.0),  # Best between 5-15
        'alpha': trial.suggest_float('alpha', 8.0, 15.0),  # Optimal range 8-15
        'n_estimators': 1500  # Increased to compensate for lower LR
    }
    # ... (rest of the objective function is the same as before) ...
    params['quantile_alpha'] = 0.05
    model_lower = xgb.XGBRegressor(**params, early_stopping_rounds=30)
    model_lower.fit(X_meta_train_tune, y_meta_train_tune, eval_set=[(X_meta_val_tune, y_meta_val_tune)], verbose=False)

    params['quantile_alpha'] = 0.95
    model_upper = xgb.XGBRegressor(**params, early_stopping_rounds=30)
    model_upper.fit(X_meta_train_tune, y_meta_train_tune, eval_set=[(X_meta_val_tune, y_meta_val_tune)], verbose=False)

    lower_preds = model_lower.predict(X_meta_val_tune)
    upper_preds = model_upper.predict(X_meta_val_tune)
    score = winkler_score(y_meta_val_tune, lower_preds, upper_preds)
    return score

# --- Run the Optuna Study ---
print(f"Running Optuna study for {N_OPTUNA_TRIALS} trials on ENHANCED features...")
study_meta = optuna.create_study(direction='minimize')
study_meta.optimize(objective_winkler_meta_enhanced, n_trials=N_OPTUNA_TRIALS)

best_params_meta = study_meta.best_params
print("\n--- Tuning Complete ---")
print(f"Best Winkler Score found during tuning: {study_meta.best_value:,.2f}")
print("Best parameters for meta-model:")
print(best_params_meta)

### **Cell 5: Train Final Meta-Model, Evaluate, and Create Submission (BUG FIXED)**

print("\n--- STAGE 3: Training Final Meta-Model & Final Showdown (BUG FIXED) ---")

# --- Create ENHANCED Meta-Features for the Test Set ---
# We must perform the same feature engineering for the test set.
df_test_raw = pd.read_csv(DATA_PATH + 'test.csv')
df_test_raw = df_test_raw.rename(columns={'latitude': 'lat', 'longitude': 'long', 'year_built': 'yr_built', 'year_reno':'yr_renovated'})
X_context_test = df_test_raw[['grade', 'sqft', 'lat', 'long', 'yr_built', 'yr_renovated']].fillna(0)
X_context_test['property_age'] = 2025 - X_context_test['yr_built']

X_meta_test_enhanced = pd.concat([
    pd.DataFrame({
        'xgb_pred': test_xgb_preds,
        'cb_pred': test_cb_preds,
        'nn_pred': test_nn_preds,
        'ensemble_mean': test_ensemble_mean
    }),
    X_context_test.reset_index(drop=True)
], axis=1)


# --- Train Final Models on ALL OOF Data ---
final_params = best_params_meta.copy()
# ***** THE BUG FIX IS HERE *****
final_params['objective'] = 'reg:quantileerror' 

# Train LOWER bound model (q=0.05)
print("\nTraining FINAL lower bound meta-model (q=0.05) on all OOF data...")
final_params['quantile_alpha'] = 0.05
model_meta_final_lower = xgb.XGBRegressor(**final_params, n_estimators=1200)
model_meta_final_lower.fit(X_meta_train_enhanced, y_true)

# Train UPPER bound model (q=0.95)
print("Training FINAL upper bound meta-model (q=0.95) on all OOF data...")
final_params['quantile_alpha'] = 0.95
model_meta_final_upper = xgb.XGBRegressor(**final_params, n_estimators=1200)
model_meta_final_upper.fit(X_meta_train_enhanced, y_true)

# --- Evaluate the New Stacked Pipeline ---
final_oof_lower = model_meta_final_lower.predict(X_meta_train_enhanced)
final_oof_upper = model_meta_final_upper.predict(X_meta_train_enhanced)
best_score_final, final_coverage, final_width = winkler_score(y_true, final_oof_lower, final_oof_upper, return_coverage=True)

# --- THE ULTIMATE PIPELINE: FINAL SHOWDOWN ---
print("\n" + "="*60)
print("     THE ENHANCED META-MODEL PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Previous Best Score (Error Model) : ${OLD_BEST_SCORE:,.2f}")
print(f"New STACKED PIPELINE Score        : ${best_score_final:,.2f}")
print(f"  - Final OOF Coverage            : {final_coverage:.2%}")
print(f"  - Final OOF Average Width       : ${final_width:,.2f}")
print("="*60)

if best_score_final < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The new context-aware pipeline is the champion!")
else:
    print("\nCONCLUSION: So close! The original pipeline remains superior.")


--- STAGE 2: Tuning the ENHANCED Quantile Stacking Meta-Model ---
Loading original raw data for feature engineering...


[I 2025-07-17 15:49:55,139] A new study created in memory with name: no-name-38cb0711-8a76-4436-b9a2-72266ca6d30e


Enhanced meta-model training data created with shape: (200000, 11)
Running Optuna study for 100 trials on ENHANCED features...


[I 2025-07-17 15:50:12,532] Trial 0 finished with value: 314426.06999953126 and parameters: {'learning_rate': 0.023432467806058312, 'max_depth': 6, 'subsample': 0.801248684706803, 'colsample_bytree': 0.8221269364787669, 'lambda': 11.661412015844993, 'alpha': 13.545003162091492}. Best is trial 0 with value: 314426.06999953126.
[I 2025-07-17 15:50:44,746] Trial 1 finished with value: 314156.9945475 and parameters: {'learning_rate': 0.020517049723537985, 'max_depth': 5, 'subsample': 0.8922522978980075, 'colsample_bytree': 0.8568477901778748, 'lambda': 9.99009015525264, 'alpha': 9.102350890531579}. Best is trial 1 with value: 314156.9945475.
[I 2025-07-17 15:51:15,425] Trial 2 finished with value: 314138.3646675 and parameters: {'learning_rate': 0.01785825030706164, 'max_depth': 5, 'subsample': 0.8609509893382344, 'colsample_bytree': 0.8025613983945264, 'lambda': 11.905083901817484, 'alpha': 9.04433209703621}. Best is trial 2 with value: 314138.3646675.
[I 2025-07-17 15:51:36,857] Trial 3 


--- Tuning Complete ---
Best Winkler Score found during tuning: 313,679.22
Best parameters for meta-model:
{'learning_rate': 0.019348815856185104, 'max_depth': 6, 'subsample': 0.8114062044495766, 'colsample_bytree': 0.8422939245056547, 'lambda': 10.191435740564742, 'alpha': 11.015101402129236}

--- STAGE 3: Training Final Meta-Model & Final Showdown (BUG FIXED) ---

Training FINAL lower bound meta-model (q=0.05) on all OOF data...
Training FINAL upper bound meta-model (q=0.95) on all OOF data...

     THE ENHANCED META-MODEL PIPELINE: FINAL SHOWDOWN
Previous Best Score (Error Model) : $296,350.26
New STACKED PIPELINE Score        : $290,890.59
  - Final OOF Coverage            : 90.14%
  - Final OOF Average Width       : $214,066.88

CONCLUSION: VICTORY! The new context-aware pipeline is the champion!


In [21]:
import joblib
import os

# --- Define a path to save the models ---
MODELS_SAVE_PATH = './final_models/'

# Create the directory if it doesn't exist
os.makedirs(MODELS_SAVE_PATH, exist_ok=True)
print(f"Directory '{MODELS_SAVE_PATH}' ensured to exist.")

# --- Save the two trained meta-models ---
lower_model_path = os.path.join(MODELS_SAVE_PATH, 'meta_model_xg_final_lower.joblib')
upper_model_path = os.path.join(MODELS_SAVE_PATH, 'meta_model_xg_final_upper.joblib')

joblib.dump(model_meta_final_lower, lower_model_path)
joblib.dump(model_meta_final_upper, upper_model_path)

print(f"\nMeta-models saved successfully!")
print(f"Lower bound model saved to: {lower_model_path}")
print(f"Upper bound model saved to: {upper_model_path}")

Directory './final_models/' ensured to exist.

Meta-models saved successfully!
Lower bound model saved to: ./final_models/meta_model_xg_final_lower.joblib
Upper bound model saved to: ./final_models/meta_model_xg_final_upper.joblib


In [22]:
# --- STAGE 3A: Tuning the CATBOOST Quantile Stacking Meta-Model ---
import catboost as cb

print("\n--- Tuning the SECOND (CatBoost) Quantile Stacking Meta-Model with Optuna ---")

# We can reuse the same enhanced feature set and validation split from before
# X_meta_train_enhanced, X_meta_val_tune, etc.

# --- Define the Optuna Objective Function for CatBoost Meta-Model ---
N_OPTUNA_TRIALS = 100
def objective_winkler_meta_cb(trial):
    """Finds the best CatBoost params for the meta-model by minimizing Winkler Score."""
    params = {
        'verbose': 0,
        'task_type': 'CPU',
        'learning_rate': trial.suggest_float('learning_rate', 0.035, 0.055),  # Narrowed around best 0.045
        'depth': trial.suggest_int('depth', 6, 8),  # Best values clustered here (6-8)
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 8.0, 12.0),  # Optimal range 8-12
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.4, 0.65),  # Best between 0.4-0.65
        'random_strength': trial.suggest_float('random_strength', 0.15, 0.35),  # Optimal range 0.15-0.35
        'iterations': 1800  # Increased to compensate for lower LR
    }
    # Train Lower Bound Model
    params['loss_function'] = 'Quantile:alpha=0.05'
    model_lower = cb.CatBoostRegressor(**params, early_stopping_rounds=30)
    model_lower.fit(X_meta_train_tune, y_meta_train_tune, eval_set=[(X_meta_val_tune, y_meta_val_tune)], verbose=False)

    # Train Upper Bound Model
    params['loss_function'] = 'Quantile:alpha=0.95'
    model_upper = cb.CatBoostRegressor(**params, early_stopping_rounds=30)
    model_upper.fit(X_meta_train_tune, y_meta_train_tune, eval_set=[(X_meta_val_tune, y_meta_val_tune)], verbose=False)
    
    lower_preds = model_lower.predict(X_meta_val_tune)
    upper_preds = model_upper.predict(X_meta_val_tune)
    score = winkler_score(y_meta_val_tune, lower_preds, upper_preds)
    return score

# --- Run the Optuna Study ---
print(f"Running Optuna study for {N_OPTUNA_TRIALS} trials for CatBoost meta-model...")
study_meta_cb = optuna.create_study(direction='minimize')
study_meta_cb.optimize(objective_winkler_meta_cb, n_trials=N_OPTUNA_TRIALS)

best_params_meta_cb = study_meta_cb.best_params
print("\n--- CatBoost Tuning Complete ---")
print(f"Best Winkler Score found during tuning: {study_meta_cb.best_value:,.2f}")

[I 2025-07-17 16:25:47,872] A new study created in memory with name: no-name-a4fe7063-b64a-46b5-9a99-c14d5fa5f201



--- Tuning the SECOND (CatBoost) Quantile Stacking Meta-Model with Optuna ---
Running Optuna study for 100 trials for CatBoost meta-model...


[I 2025-07-17 16:25:52,790] Trial 0 finished with value: 316739.70988242666 and parameters: {'learning_rate': 0.04611439488269786, 'depth': 8, 'l2_leaf_reg': 11.786821873393894, 'bagging_temperature': 0.44546455273051677, 'random_strength': 0.3223887072439535}. Best is trial 0 with value: 316739.70988242666.
[I 2025-07-17 16:26:00,218] Trial 1 finished with value: 316601.7580850309 and parameters: {'learning_rate': 0.04577262990821741, 'depth': 7, 'l2_leaf_reg': 9.927765126822052, 'bagging_temperature': 0.4331738415004225, 'random_strength': 0.2387109915653185}. Best is trial 1 with value: 316601.7580850309.
[I 2025-07-17 16:26:04,950] Trial 2 finished with value: 317428.86271623446 and parameters: {'learning_rate': 0.04667057483806934, 'depth': 8, 'l2_leaf_reg': 9.307731313704027, 'bagging_temperature': 0.4792807674215491, 'random_strength': 0.1881258262622012}. Best is trial 1 with value: 316601.7580850309.
[I 2025-07-17 16:26:13,718] Trial 3 finished with value: 316256.831644948 and


--- CatBoost Tuning Complete ---
Best Winkler Score found during tuning: 315,409.79


In [23]:
import joblib
import os
import catboost as cb

# --- Define the path to save the models ---
MODELS_SAVE_PATH = './final_models/'
os.makedirs(MODELS_SAVE_PATH, exist_ok=True)
print(f"Directory '{MODELS_SAVE_PATH}' ensured to exist.")

# --- Get the best parameters from the Optuna study ---
# This assumes 'best_params_meta_cb' exists from the previous tuning cell
final_params_cb = best_params_meta_cb.copy()

# --- 1. Train the FINAL Lower Bound Model (q=0.05) ---
print("\n--- Training FINAL CatBoost Lower Bound Model on all OOF data... ---")
final_params_cb['loss_function'] = 'Quantile:alpha=0.05'
# We create the final model object here, in the main scope
model_cb_lower = cb.CatBoostRegressor(**final_params_cb, iterations=1200, verbose=0)
model_cb_lower.fit(X_meta_train_enhanced, y_true) # Train on the full enhanced meta-feature set

# --- 2. Train the FINAL Upper Bound Model (q=0.95) ---
print("--- Training FINAL CatBoost Upper Bound Model on all OOF data... ---")
final_params_cb['loss_function'] = 'Quantile:alpha=0.95'
# We create the second final model object here
model_cb_upper = cb.CatBoostRegressor(**final_params_cb, iterations=1200, verbose=0)
model_cb_upper.fit(X_meta_train_enhanced, y_true) # Train on the full enhanced meta-feature set

print("\nFinal CatBoost meta-models for lower and upper bounds have been trained.")

# --- 3. Save the newly trained models ---
lower_cb_model_path = os.path.join(MODELS_SAVE_PATH, 'meta_model_catboost_lower.joblib')
upper_cb_model_path = os.path.join(MODELS_SAVE_PATH, 'meta_model_catboost_upper.joblib')

joblib.dump(model_cb_lower, lower_cb_model_path)
joblib.dump(model_cb_upper, upper_cb_model_path)

print(f"\nCatBoost meta-models saved successfully!")
print(f"Lower bound model saved to: {lower_cb_model_path}")
print(f"Upper bound model saved to: {upper_cb_model_path}")

Directory './final_models/' ensured to exist.

--- Training FINAL CatBoost Lower Bound Model on all OOF data... ---
--- Training FINAL CatBoost Upper Bound Model on all OOF data... ---

Final CatBoost meta-models for lower and upper bounds have been trained.

CatBoost meta-models saved successfully!
Lower bound model saved to: ./final_models/meta_model_catboost_lower.joblib
Upper bound model saved to: ./final_models/meta_model_catboost_upper.joblib


In [25]:
# --- STAGE 4 (COMPREHENSIVE): Blending Analysis and Final Showdown ---
from scipy.optimize import minimize
import pandas as pd

print("--- Starting Comprehensive Blending Analysis ---")

# --- A. Evaluate Standalone Model Performance ---
print("\n" + "="*60)
print("PART A: Evaluating the performance of each meta-model individually.")
print("="*60)

# Calculate score for XGBoost alone
winkler_xgb, coverage_xgb, width_xgb = winkler_score(y_true, oof_xgb_lower, oof_xgb_upper, return_coverage=True)
print(f"\n1. XGBoost Meta-Model OOF Score: ${winkler_xgb:,.2f}")
print(f"   - Coverage: {coverage_xgb:.2%}")
print(f"   - Avg. Width: ${width_xgb:,.2f}")

# Calculate score for CatBoost alone
winkler_cb, coverage_cb, width_cb = winkler_score(y_true, oof_cb_lower, oof_cb_upper, return_coverage=True)
print(f"\n2. CatBoost Meta-Model OOF Score: ${winkler_cb:,.2f}")
print(f"   - Coverage: {coverage_cb:.2%}")
print(f"   - Avg. Width: ${width_cb:,.2f}")


# --- B. Manual Grid Search for Blending Weights ---
print("\n" + "="*60)
print("PART B: Manually testing different blend weights to see the effect.")
print("="*60)

grid_search_results = {}
for w_xgb in np.arange(0, 1.01, 0.1): # Test weights from 0.0 to 1.0 in 10% increments
    w_cb = 1 - w_xgb
    
    # Create the blended interval predictions
    final_lower = w_xgb * oof_xgb_lower + w_cb * oof_cb_lower
    final_upper = w_xgb * oof_xgb_upper + w_cb * oof_cb_upper
    
    score = winkler_score(y_true, final_lower, final_upper)
    grid_search_results[w_xgb] = score
    
    print(f"Weight (XGB/CB): {w_xgb:.2f}/{w_cb:.2f} | Winkler Score: ${score:,.2f}")

# Find the best result from our manual search
best_manual_weight = min(grid_search_results, key=grid_search_results.get)
best_manual_score = grid_search_results[best_manual_weight]
print(f"\nBest score from manual search was ${best_manual_score:,.2f} at a weight of {best_manual_weight:.2f} for XGBoost.")


# --- C. Find the Precise Optimal Blend Weight with Optimizer ---
print("\n" + "="*60)
print("PART C: Using an optimizer to find the mathematically precise best weight.")
print("="*60)

def get_winkler_from_blend(weights):
    w_xgb = weights[0]
    w_cb = 1 - w_xgb
    final_lower = w_xgb * oof_xgb_lower + w_cb * oof_cb_lower
    final_upper = w_xgb * oof_xgb_upper + w_cb * oof_cb_upper
    return winkler_score(y_true, final_lower, final_upper)

result_blend = minimize(get_winkler_from_blend, [0.5], method='L-BFGS-B', bounds=[(0,1)])
best_xgb_weight = result_blend.x[0]
best_final_score = result_blend.fun
print(f"Optimizer found the best score of ${best_final_score:,.2f} at a weight of {best_xgb_weight:.4f} for XGBoost.")


# --- D. THE FINAL, FINAL SHOWDOWN ---
print("\n" + "="*70)
print("                  FINAL SHOWDOWN: COMPARING ALL STRATEGIES")
print("="*70)

# Create a summary DataFrame
summary = pd.DataFrame({
    'Strategy': [
        'XGBoost Meta-Model ONLY',
        'CatBoost Meta-Model ONLY',
        '50/50 Manual Blend',
        'Best Manual Blend',
        'OPTIMIZER FINAL BLEND'
    ],
    'XGB Weight': [1.0, 0.0, 0.5, best_manual_weight, best_xgb_weight],
    'Winkler Score': [
        winkler_xgb,
        winkler_cb,
        grid_search_results[0.5],
        best_manual_score,
        best_final_score
    ]
})
summary = summary.sort_values('Winkler Score').reset_index(drop=True)
print(summary.to_string())

print("\n" + "="*70)
print("\nCONCLUSION:")
if best_xgb_weight < 0.0001:
    print("The analysis is clear: the CatBoost meta-model is the superior standalone model.")
    print("Blending with the XGBoost model actually worsens the score, so we give it a weight of 0.")
elif best_xgb_weight > 0.9999:
    print("The analysis is clear: the XGBoost meta-model is the superior standalone model.")
    print("Blending with the CatBoost model actually worsens the score, so we give it a weight of 0.")
else:
    print("The analysis is clear: a blend of both models is superior to either one individually.")

--- Starting Comprehensive Blending Analysis ---

PART A: Evaluating the performance of each meta-model individually.

1. XGBoost Meta-Model OOF Score: $290,890.59
   - Coverage: 90.14%
   - Avg. Width: $214,066.88

2. CatBoost Meta-Model OOF Score: $283,164.67
   - Coverage: 90.05%
   - Avg. Width: $208,451.11

PART B: Manually testing different blend weights to see the effect.
Weight (XGB/CB): 0.00/1.00 | Winkler Score: $283,164.67
Weight (XGB/CB): 0.10/0.90 | Winkler Score: $283,616.33
Weight (XGB/CB): 0.20/0.80 | Winkler Score: $284,193.16
Weight (XGB/CB): 0.30/0.70 | Winkler Score: $284,843.42
Weight (XGB/CB): 0.40/0.60 | Winkler Score: $285,554.57
Weight (XGB/CB): 0.50/0.50 | Winkler Score: $286,315.69
Weight (XGB/CB): 0.60/0.40 | Winkler Score: $287,123.80
Weight (XGB/CB): 0.70/0.30 | Winkler Score: $287,981.75
Weight (XGB/CB): 0.80/0.20 | Winkler Score: $288,891.15
Weight (XGB/CB): 0.90/0.10 | Winkler Score: $289,853.68
Weight (XGB/CB): 1.00/0.00 | Winkler Score: $290,890.59

B

In [26]:
# --- Create Final Submission File ---
print("\n--- Creating final submission file ---")
final_test_lower = model_meta_final_lower.predict(X_meta_test_enhanced)
final_test_upper = model_meta_final_upper.predict(X_meta_test_enhanced)

submission_df = pd.DataFrame({
    'id': pd.read_csv('./test.csv')['id'],
    'pi_lower': final_test_lower,
    'pi_upper': final_test_upper
})
submission_df['pi_lower'] = submission_df['pi_lower'].clip(0, None)
submission_df['pi_upper'] = np.maximum(submission_df['pi_lower'], submission_df['pi_upper'])

submission_filename = f'submission_enhanced_meta_{int(best_score_final)}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
display(submission_df.head())


--- Creating final submission file ---

'submission_enhanced_meta_290890.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,836124.06250,1.086725e+06
1,200001,575810.93750,7.769821e+05
2,200002,462798.81250,6.367934e+05
3,200003,285117.96875,4.188179e+05
4,200004,359627.15625,7.264706e+05
